# k-NN Regression Demo

This notebook walks through k-Nearest Neighbors (k-NN) Regression on the California Housing dataset.

**Key ideas:**
- k-NN predicts targets by averaging the values of the *k* closest training points.
- Distances drive predictions, so feature **scaling is critical**.
- Hyperparameters to watch: number of neighbors (`n_neighbors`), weighting strategy, and distance metric.


## Setup
Install dependencies if needed and import the project modules.


In [ ]:
%pip install -q -r ../requirements.txt


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from app.data import load_california_housing
from app.preprocess import split_and_scale
from app.model import build_knn_regressor
from app.evaluate import regression_metrics
from app import visualize


## Load the California Housing data


In [ ]:
X, y = load_california_housing()
X.head()


## Train/test split and scaling

Because k-NN relies on distances, unscaled features would let larger-valued features dominate. Standardization gives each feature comparable influence.


In [ ]:
data = split_and_scale(X, y)
X_train, X_test, y_train, y_test = data.X_train, data.X_test, data.y_train, data.y_test
X_train.describe().head()


## Build and train the model

- **k (n_neighbors):** Small values capture local detail but may be noisy. Larger values smooth predictions.
- **weights='distance':** closer neighbors count more.
- **metric='euclidean':** standard straight-line distance.


In [ ]:
model = build_knn_regressor(n_neighbors=5, weights='distance', metric='euclidean')
model.fit(X_train, y_train)


## Evaluate


In [ ]:
preds = model.predict(X_test)
metrics = regression_metrics(y_test, preds)
metrics


## Visualize
Predicted vs Actual should ideally align along the diagonal. Residuals should center around zero without huge skew.


In [ ]:
pred_plot = visualize.plot_predictions(y_test.values, preds)
residual_plot = visualize.plot_residuals(y_test.values, preds)
pred_plot, residual_plot


## Effect of k
Try changing `n_neighbors` to see how bias/variance trade off:
- Smaller k → potentially lower bias but higher variance.
- Larger k → smoother predictions but risk underfitting.


In [ ]:
results = []
for k in [2, 5, 10, 20]:
    model_k = build_knn_regressor(n_neighbors=k, weights='distance')
    model_k.fit(X_train, y_train)
    preds_k = model_k.predict(X_test)
    m = regression_metrics(y_test, preds_k)
    results.append((k, m['rmse'], m['r2']))

for k, rmse, r2 in results:
    print(f'k={k}: RMSE={rmse:.3f}, R2={r2:.3f}')
